In [8]:
import pandas as pd
import re

train_path = "./X_train/clinical_train.csv"
test_path = "./X_test/clinical_test.csv"

def load_clinical(train_path, test_path):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    return train, test


In [9]:
import re
import pandas as pd


def parse_cytogenic(value, column_name="CYTOGENETICS"):
    """
    Parse a cytogenetic/ISCN string into structured features.

    The parser is intentionally tolerant of imperfect ISCN notation because
    the dataset contains abbreviated, malformed, and partially specified
    karyotypes.

    Parameters
    ----------
    value : str
        Cytogenetic description.

    Returns
    -------
    dict
        Structured cytogenetic features.
    """

    # ------------------------------------------------------------------
    # basic cleaning
    # ------------------------------------------------------------------
    if pd.isna(value):
        value = ""

    value = str(value).strip().lower()

    features = {
        "cyto_unknown": 0,
        "cyto_normal": 0,
        "is_mosaic": 0,
        "n_clones": 1,
        "n_abnormal_clones": 0,
        "abnormal_cell_pct": None,
        "n_cyto_events": 0,
        "is_complex": 0,
        "is_monosomal": 0,
        "has_deletion": 0,
        "has_gain": 0,
        "has_loss": 0,
        "has_translocation": 0,
        "has_inversion": 0,
        "has_duplication": 0,
        "has_additional_material": 0,
        "has_derivative": 0,
        "has_isodicentric": 0,
        "has_dicentric": 0,
        "has_ring": 0,
        "has_marker": 0,
        "has_insert": 0,
        "has_chr3_abn": 0,
        "has_inv3": 0,
        "has_t_3_3": 0,
        "has_3q21": 0,
        "has_3q26": 0,
        "has_minus5": 0,
        "has_5q_del": 0,
        "has_minus7": 0,
        "has_7q_abn": 0,
        "has_17p_abn": 0,
        "has_20q_del": 0,
        "has_chr1_abn": 0,
        "has_chr2_abn": 0,
        "has_chr4_abn": 0,
        "has_chr5_abn": 0,
        "has_chr6_abn": 0,
        "has_chr7_abn": 0,
        "has_chr8_abn": 0,
        "has_chr9_abn": 0,
        "has_chr10_abn": 0,
        "has_chr11_abn": 0,
        "has_chr12_abn": 0,
        "has_chr13_abn": 0,
        "has_chr14_abn": 0,
        "has_chr15_abn": 0,
        "has_chr16_abn": 0,
        "has_chr17_abn": 0,
        "has_chr18_abn": 0,
        "has_chr19_abn": 0,
        "has_chr20_abn": 0,
        "has_chr21_abn": 0,
        "has_chr22_abn": 0,
        "has_chrX_abn": 0,
        "has_chrY_abn": 0,
    }

    # ------------------------------------------------------------------
    # missing / unknown
    # ------------------------------------------------------------------
    if value in {
        "",
        "none",
        "nan",
        "na",
        "n/a",
        "unknown",
        "complex/other",
    }:
        if value == "complex/other":
            features["is_complex"] = 1
        else:
            features["cyto_unknown"] = 1

        return features

    # ------------------------------------------------------------------
    # normal karyotype
    # ------------------------------------------------------------------
    # Examples:
    #   46,xx
    #   46,xy
    #   46,x,-y is NOT normal
    # ------------------------------------------------------------------
    if re.fullmatch(r"\s*46\s*,\s*(xx|xy)\s*", value):
        features["cyto_normal"] = 1
        return features

    # ------------------------------------------------------------------
    # mosaic detection
    #
    # Example:
    #   46,XX,del(5)(q13q33)[10]/46,XX[5]
    # ------------------------------------------------------------------
    if "/" in value:
        features["is_mosaic"] = 1

        clones = [c.strip() for c in value.split("/") if c.strip()]
        features["n_clones"] = len(clones)

        abnormal_clones = 0
        abnormal_cells = 0
        total_cells = 0

        for clone in clones:
            # cell count: [10]
            count_match = re.search(r"\[(\d+)\]\s*$", clone)

            count = 1
            if count_match:
                count = int(count_match.group(1))

            total_cells += count

            # remove clone count before checking abnormalities
            clone_body = re.sub(r"\[\d+\]\s*$", "", clone)

            # a clone is abnormal if anything follows the sex chromosome
            # specification.
            abnormal = bool(
                re.search(
                    r"(del|add|dup|inv|t\(|der|dic|idic|i\(|r\(|-|\+mar|\+"
                    r"|hsr|ins)",
                    clone_body,
                )
            )

            if abnormal:
                abnormal_clones += 1
                abnormal_cells += count

        features["n_abnormal_clones"] = abnormal_clones

        if total_cells > 0:
            features["abnormal_cell_pct"] = (
                abnormal_cells / total_cells
            )

    # ------------------------------------------------------------------
    # generic cytogenetic events
    # ------------------------------------------------------------------

    features["has_deletion"] = int(bool(re.search(r"\bdel\s*\(", value)))
    features["has_gain"] = int(bool(re.search(r"\+(?:\d+|mar)", value)))
    features["has_loss"] = int(bool(re.search(r"(?<!\w)-\d+", value)))
    features["has_translocation"] = int(bool(re.search(r"\bt\s*\(", value)))
    features["has_inversion"] = int(bool(re.search(r"\binv\s*\(", value)))
    features["has_duplication"] = int(bool(re.search(r"\bdup\s*\(", value)))
    features["has_additional_material"] = int(bool(re.search(r"\badd\s*\(", value)))
    features["has_derivative"] = int(bool(re.search(r"\bder", value)))
    features["has_isodicentric"] = int(bool(re.search(r"\bidic", value)))
    features["has_dicentric"] = int(bool(re.search(r"\bdic", value)))
    features["has_ring"] = int(bool(re.search(r"\br\(", value)))
    features["has_marker"] = int(bool(re.search(r"\bmar\d*", value)))
    features["has_insert"] = int(bool(re.search(r"\bins\s*\(", value)))

    # ------------------------------------------------------------------
    # chromosome-level abnormalities
    # ------------------------------------------------------------------

    for chrom in range(1, 23):
        pattern = rf"(?<!\d){chrom}(?!\d)"

        # only mark chromosome if it appears as part of a cytogenetic
        # abnormality, rather than merely in a cell-count like 46.
        if re.search(
            rf"(del|add|dup|inv|t|der|dic|idic|i|r|ins)\s*\([^)]*"
            rf"{chrom}"
            rf"|(?:^|[,\s])[-+]{chrom}(?:[,/\s]|$)",
            value,
        ):
            features[f"has_chr{chrom}_abn"] = 1

    # explicit chromosome losses/gains
    for chrom in range(1, 23):
        if re.search(rf"(?:^|[,/\s])-({chrom})(?:[,/\s]|$)", value):
            features[f"has_chr{chrom}_abn"] = 1

        if re.search(rf"(?:^|[,/\s])\+({chrom})(?:[,/\s]|$)", value):
            features[f"has_chr{chrom}_abn"] = 1

    if re.search(r"(?:^|[,/\s])[-+]x(?:[,/\s]|$)", value):
        features["has_chrX_abn"] = 1

    if re.search(r"(?:^|[,/\s])[-+]y(?:[,/\s]|$)", value):
        features["has_chrY_abn"] = 1

    # ------------------------------------------------------------------
    # chromosome 3
    # ------------------------------------------------------------------

    # any event involving chromosome 3
    if re.search(
        r"(?:del|add|dup|inv|t|der|dic|idic|i|r|ins)\s*\([^)]*3"
        r"|(?:^|[,/\s])[-+]3(?:[,/\s]|$)",
        value,
    ):
        features["has_chr3_abn"] = 1

    # inv(3)
    if re.search(r"\binv\s*\(\s*3\s*\)", value):
        features["has_inv3"] = 1
        features["has_chr3_abn"] = 1

    # t(3;3)
    if re.search(r"\bt\s*\(\s*3\s*;\s*3\s*\)", value):
        features["has_t_3_3"] = 1
        features["has_chr3_abn"] = 1

    # 3q21 / 3q26
    if re.search(r"3\s*q\s*21", value):
        features["has_3q21"] = 1
        features["has_chr3_abn"] = 1

    if re.search(r"3\s*q\s*26", value):
        features["has_3q26"] = 1
        features["has_chr3_abn"] = 1

    # ------------------------------------------------------------------
    # important AML abnormalities
    # ------------------------------------------------------------------

    # monosomy 5 / chromosome 5 deletion
    if re.search(r"(?:^|[,/\s])-5(?:[,/\s]|$)", value):
        features["has_minus5"] = 1

    if re.search(r"\bdel\s*\(\s*5\s*\)", value):
        features["has_5q_del"] = 1
        features["has_chr5_abn"] = 1

    # monosomy 7
    if re.search(r"(?:^|[,/\s])-7(?:[,/\s]|$)", value):
        features["has_minus7"] = 1
        features["has_chr7_abn"] = 1

    # 7q abnormality
    if re.search(r"7\s*q", value):
        features["has_7q_abn"] = 1
        features["has_chr7_abn"] = 1

    # 17p
    if re.search(r"17\s*p", value):
        features["has_17p_abn"] = 1
        features["has_chr17_abn"] = 1

    # 20q deletion
    if re.search(r"\bdel\s*\(\s*20\s*\)\s*\(\s*q", value):
        features["has_20q_del"] = 1
        features["has_chr20_abn"] = 1

    # ------------------------------------------------------------------
    # count cytogenetic events
    # ------------------------------------------------------------------

    event_patterns = [
        r"\bdel\s*\(",
        r"\badd\s*\(",
        r"\bdup\s*\(",
        r"\binv\s*\(",
        r"\bt\s*\(",
        r"\bder",
        r"\bdic",
        r"\bidic",
        r"\bi\s*\(",
        r"\br\s*\(",
        r"\bins\s*\(",
        r"(?<!\w)-\d+",
        r"(?<!\w)\+\d+",
        r"\+mar",
    ]

    event_count = 0

    for pattern in event_patterns:
        event_count += len(re.findall(pattern, value))

    features["n_cyto_events"] = event_count

    # ------------------------------------------------------------------
    # monosomal karyotype
    #
    # basic operational definition:
    # >=2 autosomal chromosome losses OR
    # one autosomal loss + structural abnormality.
    #
    # keep this deliberately transparent rather than pretending to
    # reproduce a full clinical classification algorithm.
    # ------------------------------------------------------------------

    monosomies = re.findall(r"(?:^|[,/\s])-(\d+)(?:[,/\s]|$)", value)

    if len(set(monosomies)) >= 2:
        features["is_monosomal"] = 1

    elif len(set(monosomies)) >= 1 and (
        features["has_deletion"]
        or features["has_translocation"]
        or features["has_inversion"]
        or features["has_derivative"]
    ):
        features["is_monosomal"] = 1

    # ------------------------------------------------------------------
    # complex karyotype
    #
    # use number of detected cytogenetic events rather than trying to
    # interpret every possible ISCN construction.
    # ------------------------------------------------------------------

    if features["n_cyto_events"] >= 3:
        features["is_complex"] = 1

    # Explicit label
    if "complex" in value:
        features["is_complex"] = 1

    return features

In [10]:
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

def preprocess_clinical(train, test, cyto_col="CYTOGENETICS"):
    """
    Transforms clinical features and parses cytogenetic text representations 
    into clean, numeric tables optimized for tree survival estimators.
    """
    # Prevent changing original memory frames
    train_df = train.copy()
    test_df = test.copy()

    # 1. Map text parsing across datasets
    train_features = train_df[cyto_col].apply(lambda x: parse_cytogenic(x, column_name=cyto_col))
    test_features = test_df[cyto_col].apply(lambda x: parse_cytogenic(x, column_name=cyto_col))
    
    train_parsed_df = pd.DataFrame(list(train_features), index=train_df.index)
    test_parsed_df = pd.DataFrame(list(test_features), index=test_df.index)
    
    # Drop original text column and merge parsed metrics
    train_df = train_df.drop(columns=[cyto_col]).join(train_parsed_df)
    test_df = test_df.drop(columns=[cyto_col]).join(test_parsed_df)

    # 2. Treat missing numerical metadata (e.g., cell percentages) safely for trees
    # Tree frameworks handle arbitrary constants well; -1 isolates missing records distinctly
    if "abnormal_cell_pct" in train_df.columns:
        train_df["abnormal_cell_pct"] = train_df["abnormal_cell_pct"].fillna(-1.0)
        test_df["abnormal_cell_pct"] = test_df["abnormal_cell_pct"].fillna(-1.0)

    # 3. Handle Other Object/Categorical Variables
    categorical_cols = train_df.select_dtypes(include=["object", "category"]).columns.tolist()
    
    # Exclude survival outcomes if they happen to be text-typed
    survival_targets = ["time", "status", "os_months", "os_status", "event"]
    categorical_cols = [c for c in categorical_cols if c.lower() not in survival_targets]

    if categorical_cols:
        # Ordinal encoding maps text safely into high-performance integer arrays
        # unknown values encountered in test deployment route cleanly into a constant flag (-1)
        encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value", 
            unknown_value=-1, 
            encoded_missing_value=-1
        )
        
        train_df[categorical_cols] = encoder.fit_transform(train_df[categorical_cols].astype(str))
        test_df[categorical_cols] = encoder.transform(test_df[categorical_cols].astype(str))

    return train_df, test_df
